<a href="https://colab.research.google.com/github/rounak393/clab/blob/main/robertresnet80_10_10.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# IMPORTANT: SOME KAGGLE DATA SOURCES ARE PRIVATE
# RUN THIS CELL IN ORDER TO IMPORT YOUR KAGGLE DATA SOURCES.
import kagglehub
kagglehub.login()


In [ ]:
# IMPORTANT: RUN THIS CELL IN ORDER TO IMPORT YOUR KAGGLE DATA SOURCES,
# THEN FEEL FREE TO DELETE THIS CELL.
# NOTE: THIS NOTEBOOK ENVIRONMENT DIFFERS FROM KAGGLE'S PYTHON
# ENVIRONMENT SO THERE MAY BE MISSING LIBRARIES USED BY YOUR
# NOTEBOOK.

britikak_busi_dataset_path = kagglehub.dataset_download('britikak/busi-dataset')

print('Data source import complete.')


In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
from tqdm import tqdm
import random
import copy

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, Subset
import albumentations as A
import cv2
import warnings
import torch.nn.functional as F
from torchinfo import summary
from torchvision.models import resnet34, ResNet34_Weights

warnings.filterwarnings("ignore")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

BASE_DIR = "/kaggle/input/datasets/britikak/busi-dataset/Dataset_BUSI_with_GT"
CLASSES = ["benign", "malignant"]
IMG_SIZE = 256
BATCH_SIZE = 8
SEED = 42

torch.manual_seed(SEED)
random.seed(SEED)
np.random.seed(SEED)
torch.backends.cudnn.benchmark = True
torch.backends.cudnn.deterministic = True

class BUSISegmentationDataset(Dataset):
    def __init__(self, base_dir, classes, transform=None):
        self.samples = []
        self.transform = transform

        for cls in classes:
            cls_dir = os.path.join(base_dir, cls)

            images = [
                f for f in os.listdir(cls_dir)
                if f.endswith(".png") and "_mask" not in f
            ]

            for img_name in images:
                img_path = os.path.join(cls_dir, img_name)
                base_name = img_name.replace(".png", "")

                mask_files = [
                    f for f in os.listdir(cls_dir)
                    if f.startswith(base_name + "_mask") and f.endswith(".png")
                ]

                if len(mask_files) == 0:
                    continue

                mask_paths = [os.path.join(cls_dir, f) for f in mask_files]
                self.samples.append((img_path, mask_paths))

        assert len(self.samples) > 0, "No valid image-mask pairs found!"

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        img_path, mask_paths = self.samples[idx]

        image = np.array(Image.open(img_path).convert("RGB"))

        combined_mask = np.zeros(image.shape[:2], dtype=np.uint8)
        for mpath in mask_paths:
            mask = np.array(Image.open(mpath).convert("L"))
            mask = (mask > 0).astype(np.uint8)
            combined_mask = np.logical_or(combined_mask, mask)

        combined_mask = combined_mask.astype(np.float32)

        if self.transform:
            augmented = self.transform(image=image, mask=combined_mask)
            image = augmented["image"]
            combined_mask = augmented["mask"]

        # Ensure binary mask (important!)
        combined_mask = (combined_mask > 0.5).astype(np.float32)

        image = torch.from_numpy(image).permute(2, 0, 1).float()
        mask = torch.from_numpy(combined_mask).unsqueeze(0).float()

        return image, mask

# ===================---------------------=======================
train_transform = A.Compose([
    A.Resize(IMG_SIZE, IMG_SIZE),
    A.HorizontalFlip(p=0.5),
    A.RandomRotate90(p=0.5),
    A.RandomBrightnessContrast(p=0.5),
    A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225), max_pixel_value=255.0)
])

val_transform = A.Compose([
    A.Resize(IMG_SIZE, IMG_SIZE),
    A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225), max_pixel_value=255.0)
])

test_transform = val_transform

# ===================------------------------==================
full_dataset = BUSISegmentationDataset(BASE_DIR, classes=CLASSES, transform=None)
total_size = len(full_dataset)
indices = list(range(total_size))
np.random.seed(SEED)
np.random.shuffle(indices)

train_size = int(0.8 * total_size)
val_size   = int(0.1 * total_size)

val_indices = indices[:val_size]
train_indices   = indices[val_size:train_size + val_size]
test_indices  = indices[train_size + val_size:]

train_dataset = torch.utils.data.Subset(
    BUSISegmentationDataset(BASE_DIR, CLASSES, transform=train_transform),
    train_indices)

val_dataset = torch.utils.data.Subset(
    BUSISegmentationDataset(BASE_DIR, CLASSES, transform=val_transform),
    val_indices)

test_dataset = torch.utils.data.Subset(
    BUSISegmentationDataset(BASE_DIR, CLASSES, transform=test_transform),
    test_indices)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=2,
    pin_memory=True)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    pin_memory=True)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    pin_memory=True)

# ==========================================
# ADVANCED ARCHITECTURE (MAX DICE UNET)
# ==========================================
class RobertsEdgeOperator(nn.Module):
    def __init__(self):
        super().__init__()
        kx = torch.tensor([[[[1.0, 0.0], [0.0, -1.0]]]])
        ky = torch.tensor([[[[0.0, 1.0], [-1.0, 0.0]]]])
        self.register_buffer("kx", kx)
        self.register_buffer("ky", ky)

    def forward(self, x):
        gray = (0.2989 * x[:, 0:1] + 0.5870 * x[:, 1:2] + 0.1140 * x[:, 2:3])
        gray_padded = F.pad(gray, (0, 1, 0, 1), mode="replicate")
        gx   = F.conv2d(gray_padded, self.kx)
        gy   = F.conv2d(gray_padded, self.ky)
        return torch.sqrt(gx ** 2 + gy ** 2 + 1e-8)

class PCBAM_Filter(nn.Module):
    def __init__(self, in_channels, reduction=8):
        super().__init__()
        self.cam = nn.Sequential(
            nn.Linear(in_channels, in_channels // reduction),
            nn.ReLU(inplace=True),
            nn.Linear(in_channels // reduction, in_channels)
        )
        self.sam = nn.Conv2d(2, 1, kernel_size=7, padding=3, bias=False)

    def forward(self, x):
        b, c, _, _ = x.size()
        avg = self.cam(F.adaptive_avg_pool2d(x, 1).view(b, c)).view(b, c, 1, 1)
        mxp = self.cam(F.adaptive_max_pool2d(x, 1).view(b, c)).view(b, c, 1, 1)
        x_c = x * torch.sigmoid(avg + mxp)
        sp  = torch.cat([
            torch.mean(x_c, dim=1, keepdim=True),
            torch.max(x_c,  dim=1, keepdim=True)[0]
        ], dim=1)
        return x_c * torch.sigmoid(self.sam(sp))

class SpatialGateAttention(nn.Module):
    def __init__(self, in_channels):
        super().__init__()
        mid = in_channels // 8
        self.q_conv   = nn.Conv2d(in_channels, mid, 1)
        self.k_conv   = nn.Conv2d(in_channels, mid, 1)
        self.v_conv   = nn.Conv2d(in_channels, in_channels, 1)
        self.gate     = nn.Conv2d(mid * 2 + in_channels, in_channels, 1)

    def forward(self, x):
        q, k, v = self.q_conv(x), self.k_conv(x), self.v_conv(x)
        return x + v * torch.sigmoid(self.gate(torch.cat([q, k, v], dim=1)))

class DoubleConv(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_channels, out_channels, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True)
        )
    def forward(self, x):
        return self.block(x)

class MaxDiceUNet(nn.Module):
    def __init__(self, n_classes=1):
        super().__init__()
        self.roberts = RobertsEdgeOperator()
        resnet = resnet34(weights=ResNet34_Weights.IMAGENET1K_V1)

        # 4-channel input modification
        self.conv1 = nn.Conv2d(4, 64, kernel_size=7, stride=2, padding=3, bias=False)
        with torch.no_grad():
            self.conv1.weight[:, :3] = resnet.conv1.weight
            self.conv1.weight[:, 3]  = resnet.conv1.weight.mean(dim=1)

        self.bn1     = resnet.bn1
        self.relu    = resnet.relu
        self.maxpool = resnet.maxpool

        self.layer1 = resnet.layer1
        self.layer2 = resnet.layer2
        self.layer3 = resnet.layer3
        self.layer4 = resnet.layer4

        self.pcbam1 = PCBAM_Filter(64)
        self.pcbam2 = PCBAM_Filter(128)
        self.pcbam3 = PCBAM_Filter(256)
        self.pcbam4 = PCBAM_Filter(512)

        self.pool       = nn.MaxPool2d(2)
        self.bottleneck = DoubleConv(512, 1024)
        self.sga        = SpatialGateAttention(1024)

        self.up4  = nn.ConvTranspose2d(1024, 512, 2, stride=2)
        self.dec4 = DoubleConv(1024, 512)
        self.up3  = nn.ConvTranspose2d(512, 256, 2, stride=2)
        self.dec3 = DoubleConv(512, 256)
        self.up2  = nn.ConvTranspose2d(256, 128, 2, stride=2)
        self.dec2 = DoubleConv(256, 128)
        self.up1  = nn.ConvTranspose2d(128, 64, 2, stride=2)
        self.dec1 = DoubleConv(128, 64)
        self.up0  = nn.ConvTranspose2d(64, 64, 2, stride=2)
        self.dec0 = DoubleConv(128, 64)

        self.up_out  = nn.ConvTranspose2d(64, 32, 2, stride=2)
        self.dec_out = DoubleConv(32, 32)
        self.final   = nn.Conv2d(32, n_classes, 1)

    def forward(self, x):
        x_edge  = self.roberts(x)
        x_fused = torch.cat([x, x_edge], dim=1)

        x0 = self.relu(self.bn1(self.conv1(x_fused)))
        x1 = self.maxpool(x0)

        e1 = self.layer1(x1)
        e2 = self.layer2(e1)
        e3 = self.layer3(e2)
        e4 = self.layer4(e3)

        s1 = self.pcbam1(e1)
        s2 = self.pcbam2(e2)
        s3 = self.pcbam3(e3)
        s4 = self.pcbam4(e4)

        b = self.sga(self.bottleneck(self.pool(e4)))

        d4 = self.dec4(torch.cat([self.up4(b),  s4], dim=1))
        d3 = self.dec3(torch.cat([self.up3(d4), s3], dim=1))
        d2 = self.dec2(torch.cat([self.up2(d3), s2], dim=1))
        d1 = self.dec1(torch.cat([self.up1(d2), s1], dim=1))

        d0  = self.dec0(torch.cat([self.up0(d1), x0], dim=1))
        out = self.dec_out(self.up_out(d0))

        # RETURNS RAW LOGITS (Required for HybridLoss)
        return self.final(out)

# ==========================================
# HYBRID LOSS & METRICS
# ==========================================
class HybridLoss(nn.Module):
    def __init__(self, smooth=1.0, focal_gamma=2.0):
        super().__init__()
        self.smooth      = smooth
        self.focal_gamma = focal_gamma

    def forward(self, logits, targets):
        bce   = F.binary_cross_entropy_with_logits(logits, targets)
        probs = torch.sigmoid(logits).view(-1)
        tgt   = targets.view(-1)
        inter = (probs * tgt).sum()
        dice  = 1.0 - (2.0 * inter + self.smooth) / (probs.sum() + tgt.sum() + self.smooth)
        pt    = torch.where(tgt == 1, probs, 1.0 - probs)
        focal = (-(1.0 - pt) ** self.focal_gamma * torch.log(pt + 1e-8)).mean()
        return 0.4 * bce + 0.4 * dice + 0.2 * focal

# Dice Metric
def dice_coef(y_true, y_pred, smooth=1e-5):
    y_true_f = y_true.view(-1)
    y_pred_f = y_pred.view(-1)
    inter = (y_true_f * y_pred_f).sum()
    return (2. * inter + smooth) / (y_true_f.sum() + y_pred_f.sum() + smooth)

# IoU Metric
def iou_score(preds, masks, threshold=0.5, eps=1e-6):
    preds_bin = (preds > threshold).float()
    masks_bin = (masks > threshold).float()

    inter = (preds_bin * masks_bin).sum().item()
    union = preds_bin.sum().item() + masks_bin.sum().item() - inter

    return inter / (union + eps)

# ==========================================
# TRAINING FUNCTION
# ==========================================
def train_model(model, train_loader, val_loader, epochs=50):
    device = "cuda" if torch.cuda.is_available() else "cpu"
    model = model.to(device)

    # REPLACED BCEDiceLoss WITH HybridLoss
    criterion = HybridLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)

    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer,
        mode='min',          # because we monitor val_loss
        factor=0.5,          # reduce LR by half
        patience=10,         # wait 10 epochs before reducing,
        min_lr=1e-6
    )
    # Tracking curves
    train_losses, val_losses = [], []
    train_dice_scores, val_dice_scores = [], []
    train_iou_scores, val_iou_scores = [], []

    best_val_loss = float("inf")
    best_epoch = 0
    best_model_weights = None

    for epoch in range(epochs):
        model.train()
        train_loss = train_dice = train_iou = 0
        for images, masks in tqdm(train_loader, desc=f"Epoch {epoch+1}/{epochs} [Train]"):
            images, masks = images.to(device), masks.to(device)

            # MODEL OUTPUTS LOGITS NOW
            preds_logits = model(images)
            loss = criterion(preds_logits, masks)

            optimizer.zero_grad()
            loss.backward()

            # Gradient clipping to prevent inf loss
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)

            optimizer.step()

            # Convert logits to probabilities for metric calculation
            preds_probs = torch.sigmoid(preds_logits)

            # Accumulate
            train_loss += loss.item()
            train_dice += dice_coef(masks, preds_probs).item()
            train_iou += iou_score(preds_probs, masks)

        # ------------------ VALIDATION ------------------
        model.eval()
        val_loss = val_dice = val_iou = 0

        with torch.no_grad():
            for images, masks in tqdm(val_loader, desc=f"Epoch {epoch+1}/{epochs} [Val]"):
                images, masks = images.to(device), masks.to(device)

                preds_logits = model(images)
                loss = criterion(preds_logits, masks)

                preds_probs = torch.sigmoid(preds_logits)

                val_loss += loss.item()
                val_dice += dice_coef(masks, preds_probs).item()
                val_iou += iou_score(preds_probs, masks)

        # ------------------ AVERAGES ------------------
        avg_train_loss = train_loss / len(train_loader)
        avg_val_loss   = val_loss / len(val_loader)
        avg_train_dice = train_dice / len(train_loader)
        avg_val_dice   = val_dice / len(val_loader)
        avg_train_iou  = train_iou / len(train_loader)
        avg_val_iou    = val_iou / len(val_loader)

        train_losses.append(avg_train_loss)
        val_losses.append(avg_val_loss)
        train_dice_scores.append(avg_train_dice)
        val_dice_scores.append(avg_val_dice)
        train_iou_scores.append(avg_train_iou)
        val_iou_scores.append(avg_val_iou)

        # ------------------ PRINT ------------------
        print(f"\nEpoch {epoch+1}/{epochs}")
        print(f"Train Loss: {avg_train_loss:.4f} | Val Loss: {avg_val_loss:.4f}")
        print(f"Train Dice: {avg_train_dice:.4f} | Val Dice: {avg_val_dice:.4f}")
        print(f"Train IoU : {avg_train_iou:.4f} | Val IoU : {avg_val_iou:.4f}")

        # ------------------ SAVE BEST ------------------
        if avg_val_loss < best_val_loss:
            best_val_loss = avg_val_loss
            best_epoch = epoch + 1
            best_model_weights = copy.deepcopy(model.state_dict())
            torch.save(model.state_dict(), "best_maxdice_unet.pth")
            print(f"✓ Best model saved (Epoch {epoch+1}, Val Loss {best_val_loss:.4f})")

        scheduler.step(avg_val_loss)

    # ------------------ LOAD BEST MODEL ------------------
    model.load_state_dict(best_model_weights)

    print("\n" + "="*60)
    print("Training Completed!")
    print(f"Best Epoch: {best_epoch}")
    print(f"Best Val Loss: {best_val_loss:.4f}")
    print("="*60)

    return {
        "train_losses": train_losses,
        "val_losses": val_losses,
        "train_dice": train_dice_scores,
        "val_dice": val_dice_scores,
        "train_iou": train_iou_scores,
        "val_iou": val_iou_scores,
    }

def plot_training(history):
    plt.figure(figsize=(17, 4))

    # Loss
    plt.subplot(1, 3, 1)
    plt.plot(history["train_losses"], label="Train Loss", linewidth=2)
    plt.plot(history["val_losses"], label="Val Loss", linewidth=2)
    plt.title("Loss Curve", fontsize=18, fontweight='bold')
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.legend()
    plt.grid(True, alpha=0.3)

    # Dice
    plt.subplot(1, 3, 2)
    plt.plot(history["train_dice"], label="Train Dice", linewidth=2)
    plt.plot(history["val_dice"], label="Val Dice", linewidth=2)
    plt.title("Dice Coefficient Curve", fontsize=18, fontweight='bold')
    plt.xlabel("Epoch")
    plt.ylabel("Dice Score")
    plt.legend()
    plt.grid(True, alpha=0.3)

    # IoU
    plt.subplot(1, 3, 3)
    plt.plot(history["train_iou"], label="Train IoU", linewidth=2)
    plt.plot(history["val_iou"], label="Val IoU", linewidth=2)
    plt.title("IoU Curve", fontsize=18, fontweight='bold')
    plt.xlabel("Epoch")
    plt.ylabel("IoU Score")
    plt.legend()
    plt.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.savefig("training_curves.png", dpi=300, bbox_inches='tight')
    plt.show()

# ==========================================
# EXECUTION
# ==========================================
if __name__ == "__main__":
    model = MaxDiceUNet(n_classes=1)
    model = model.to(device)

    print("\n================= MODEL SUMMARY =================\n")
    print(summary(model, input_size=(BATCH_SIZE, 3, IMG_SIZE, IMG_SIZE)))

    history = train_model(model, train_loader, val_loader, epochs=100)
    plot_training(history)

    def iou_coef(y_true, y_pred, smooth=1e-5):
        y_true_f = y_true.view(-1)
        y_pred_f = y_pred.view(-1)
        intersection = (y_true_f * y_pred_f).sum()
        union = y_true_f.sum() + y_pred_f.sum() - intersection
        return (intersection + smooth) / (union + smooth)

    print("Loading best model...")
    model.load_state_dict(torch.load("best_maxdice_unet.pth", map_location=device))
    model.eval()

    test_dice, test_iou = 0, 0
    print("Evaluating on test set...")
    with torch.no_grad():
        for images, masks in tqdm(test_loader, desc="Evaluating Test Set"):
            images, masks = images.to(device), masks.to(device)

            # ADDED SIGMOID HERE for raw logits output
            preds_probs = torch.sigmoid(model(images))
            preds_bin = (preds_probs > 0.4).float()

            test_dice += dice_coef(masks, preds_bin).item()
            test_iou += iou_coef(masks, preds_bin).item()

    num_batches = len(test_loader)
    avg_test_dice = test_dice / num_batches
    avg_test_iou = test_iou / num_batches

    print("\n" + "="*40)
    print("TEST SET EVALUATION")
    print("="*40)
    print(f"Dice Coefficient: {avg_test_dice:.4f}")
    print(f"IoU (Jaccard):    {avg_test_iou:.4f}")
    print("="*40)

    def dice_coef_single(y_true, y_pred, smooth=1e-5):
        y_true_f = y_true.flatten()
        y_pred_f = y_pred.flatten()
        intersection = np.sum(y_true_f * y_pred_f)
        return (2. * intersection + smooth) / (np.sum(y_true_f) + np.sum(y_pred_f) + smooth)

    def iou_coef_single(y_true, y_pred, smooth=1e-5):
        intersection = np.sum(y_true * y_pred)
        union = np.sum(y_true) + np.sum(y_pred) - intersection
        return intersection / (union + smooth)

    def visualize_predictions_simple(model, dataset, num_samples=5, threshold=0.5):
        indices = random.sample(range(len(dataset)), num_samples)
        fig, axes = plt.subplots(num_samples, 4, figsize=(16, num_samples * 4))

        if num_samples == 1:
            axes = axes.reshape(1, -1)

        for i, idx in enumerate(indices):
            image, mask = dataset[idx]

            with torch.no_grad():
                # ADDED SIGMOID HERE
                pred_prob = torch.sigmoid(model(image.unsqueeze(0).to(device))).cpu().squeeze().numpy()

            pred_mask = (pred_prob > threshold).astype(np.uint8)
            true_mask = mask.squeeze().numpy().astype(np.uint8)

            dice_score = dice_coef_single(true_mask, pred_mask) * 100
            iou_score = iou_coef_single(true_mask, pred_mask) * 100

            error_map = np.zeros((*true_mask.shape, 3), dtype=np.uint8)
            true_positive = (true_mask == 1) & (pred_mask == 1)
            false_positive = (true_mask == 0) & (pred_mask == 1)
            false_negative = (true_mask == 1) & (pred_mask == 0)

            error_map[true_positive] = [255, 255, 255]
            error_map[false_positive] = [255, 0, 0]
            error_map[false_negative] = [0, 0, 255]

            axes[i, 0].imshow(image.permute(1, 2, 0) if image.shape[0] == 3 else image.squeeze(), cmap='gray')
            axes[i, 0].set_title(f"Input Image {idx}")
            axes[i, 0].axis("off")

            axes[i, 1].imshow(true_mask, cmap="gray")
            axes[i, 1].set_title("Ground Truth")
            axes[i, 1].axis("off")

            axes[i, 2].imshow(pred_mask, cmap="gray")
            axes[i, 2].set_title(f"Prediction\nDice: {dice_score:.1f}% | IoU: {iou_score:.1f}%")
            axes[i, 2].axis("off")

            axes[i, 3].imshow(error_map)
            axes[i, 3].set_title("Error Analysis\n(Red: FP, Blue: FN)")
            axes[i, 3].axis("off")

            if i == 0:
                from matplotlib.patches import Patch
                legend_elements = [
                    Patch(facecolor='white', label='True Positive'),
                    Patch(facecolor='red', label='False Positive'),
                    Patch(facecolor='blue', label='False Negative')
                ]
                axes[i, 3].legend(handles=legend_elements, loc='upper right', fontsize=8)

        plt.tight_layout()
        plt.show()

        return indices

    print("\nGenerating visualizations...")
    sampled_indices = visualize_predictions_simple(model, test_dataset, num_samples=7)
    print("\nEvaluation completed successfully!")

Using device: cuda

================= MODEL SUMMARY =================

Layer (type:depth-idx)                   Output Shape              Param #
MaxDiceUNet                              [8, 1, 256, 256]          --
├─RobertsEdgeOperator: 1-1               [8, 1, 256, 256]          --
├─Conv2d: 1-2                            [8, 64, 128, 128]         12,544
├─BatchNorm2d: 1-3                       [8, 64, 128, 128]         128
├─ReLU: 1-4                              [8, 64, 128, 128]         --
├─MaxPool2d: 1-5                         [8, 64, 64, 64]           --
├─Sequential: 1-6                        [8, 64, 64, 64]           --
│    └─BasicBlock: 2-1                   [8, 64, 64, 64]           --
│    │    └─Conv2d: 3-1                  [8, 64, 64, 64]           36,864
│    │    └─BatchNorm2d: 3-2             [8, 64, 64, 64]           128
│    │    └─ReLU: 3-3                    [8, 64, 64, 64]           --
│    │    └─Conv2d: 3-4                  [8, 64, 64, 64]           36,864


Epoch 1/100 [Val]: 100%|██████████| 8/8 [00:00<00:00, 11.25it/s]



Epoch 1/100
Train Loss: 0.5860 | Val Loss: 0.5133
Train Dice: 0.2159 | Val Dice: 0.2285
Train IoU : 0.3158 | Val IoU : 0.5471
✓ Best model saved (Epoch 1, Val Loss 0.5133)


Epoch 2/100 [Val]: 100%|██████████| 8/8 [00:00<00:00, 11.12it/s]



Epoch 2/100
Train Loss: 0.5044 | Val Loss: 0.4843
Train Dice: 0.2642 | Val Dice: 0.2531
Train IoU : 0.4862 | Val IoU : 0.5545
✓ Best model saved (Epoch 2, Val Loss 0.4843)


...


Epoch 100/100
Train Loss: 0.0457 | Val Loss: 0.2235
Train Dice: 0.9232 | Val Dice: 0.7229
Train IoU : 0.9037 | Val IoU : 0.6056

Training Completed!
Best Epoch: 62
Best Val Loss: 0.1765
Loading best model...
Evaluating on test set...


Evaluating Test Set: 100%|██████████| 9/9 [00:02<00:00,  4.32it/s]



TEST SET EVALUATION
Dice Coefficient: 0.8006
IoU (Jaccard):    0.6801

Generating visualizations...


Clipping input data to the valid range for imshow with RGB data ([0..1] for floats or [0..255] for integers). Got range [-2.117904..2.6399999].
Clipping input data to the valid range for imshow with RGB data ([0..1] for floats or [0..255] for integers). Got range [-2.117904..2.2042701].
Clipping input data to the valid range for imshow with RGB data ([0..1] for floats or [0..255] for integers). Got range [-2.0836544..2.6399999].
Clipping input data to the valid range for imshow with RGB data ([0..1] for floats or [0..255] for integers). Got range [-2.1007793..2.6399999].
Clipping input data to the valid range for imshow with RGB data ([0..1] for floats or [0..255] for integers). Got range [-2.117904..2.6399999].
Clipping input data to the valid range for imshow with RGB data ([0..1] for floats or [0..255] for integers). Got range [-2.117904..2.6399999].
Clipping input data to the valid range for imshow with RGB data ([0..1] for floats or [0..255] for integers). Got range [-2.117904..2.


Evaluation completed successfully!
